In [1]:
%pip install "ray[default]"
%pip install python-dotenv

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.0.1 -> 26.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.0.1 -> 26.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os 
import ray 

c:\Users\ASUS\OneDrive\Desktop\MLOPs\my-first-MLOPS-project\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-08-05 22:45:44,254	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


In [3]:
import sys; sys.path.append("..")
import warnings; warnings.filterwarnings("ignore")
from dotenv import load_dotenv; load_dotenv()
%load_ext autoreload
%autoreload 2

In [4]:
if ray.is_initialized():
    ray.shutdown()
ray.init(
    num_cpus= 4,
    object_store_memory=2 * 1024 * 1024 * 1024,
    runtime_env={"env_vars": {"USE_LIBUV": "0"}}
)

2026-08-05 22:45:52,963	INFO worker.py:2003 -- Started a local Ray instance. View the dashboard at 127.0.0.1:8265 


Python version:,3.10.11
Ray version:,2.55.1
Dashboard:,http://127.0.0.1:8265


In [5]:
ray.cluster_resources()

{'node:127.0.0.1': 1.0,
 'GPU': 1.0,
 'memory': 3811074048.0,
 'node:__internal_head__': 1.0,
 'object_store_memory': 2147483648.0,
 'accelerator_type:G': 1.0,
 'CPU': 4.0}

In [ ]:
num_workers = 2
resources_per_worker = {"CPU": 1, "GPU": 1}

In [7]:
import os
from pathlib import Path

if os.path.exists("/efs"):
    EFS_DIR = f"/efs/shared_storage/MY-FIRST-MLOPS-PROJECT/{os.environ.get('Eva', 'default_user')}"
else:    
    EFS_DIR = str(Path(os.getcwd()).parent / "local_storage")
    os.makedirs(EFS_DIR, exist_ok=True)

print(f"Current active storage directory: {EFS_DIR}")

Current active storage directory: /efs/shared_storage/MY-FIRST-MLOPS-PROJECT/default_user


In [8]:
import pandas as pd 

In [9]:
dataset = "https://raw.githubusercontent.com/evasim/my-first-MLOPS-project/refs/heads/main/data/raw_dataset.csv"
df = pd.read_csv(dataset)
df.head()

,text,label
0,Wall St. Bears Claw Back Into the Black (Reute...,2
1,Carlyle Looks Toward Commercial Aerospace (Reu...,2
2,Oil and Economy Cloud Stocks' Outlook (Reuters...,2
3,Iraq Halts Oil Exports from Main Southern Pipe...,2
4,"Oil prices soar to all-time record, posing new...",2


In [10]:
from sklearn.model_selection import train_test_split

In [11]:
df.label.value_counts()

label
2    30000
3    30000
1    30000
0    30000
Name: count, dtype: int64

In [12]:
test_size = 0.2 
train_df, val_df = train_test_split(df, stratify=df.label, test_size=test_size, random_state=42)

In [13]:
train_df.label.value_counts()

label
1    24000
3    24000
0    24000
2    24000
Name: count, dtype: int64

In [14]:
val_df.label.value_counts() * int((1 - test_size) / test_size)

label
2    24000
1    24000
0    24000
3    24000
Name: count, dtype: int64

In [15]:
from collections import Counter 
import matplotlib.pyplot as plt 
import seaborn as sns; sns.set_theme()
import warnings; warnings.filterwarnings("ignore")
from wordcloud import WordCloud, STOPWORDS

In [16]:
tags = Counter(df.label)
tags.most_common()

[(2, 30000), (3, 30000), (1, 30000), (0, 30000)]

In [17]:
import json 
import nltk 
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
import re

In [18]:
nltk.download("stopwords")
words = stopwords.words("english")

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\ASUS\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [19]:
df.head()

,text,label
0,Wall St. Bears Claw Back Into the Black (Reute...,2
1,Carlyle Looks Toward Commercial Aerospace (Reu...,2
2,Oil and Economy Cloud Stocks' Outlook (Reuters...,2
3,Iraq Halts Oil Exports from Main Southern Pipe...,2
4,"Oil prices soar to all-time record, posing new...",2


In [20]:
def clean_text(text, stopwords=words):
    # change every words into lower case
    text = text.lower()

    # removing stopwords such as "is", "the" and so on
    pattern = re.compile(r'\b(' + r"|".join(words)+ r")\b\s*")
    text = pattern.sub('', text)

    text = re.sub(r"([!\"'#$%&()*\+,-./:;<=>?@\\\[\]^_`{|}~])", r" \1 ", text) # add space 
    text = re.sub("[^A-Za-z0-9]+", " ", text) # remove other than words and numbers 
    text = re.sub(" +", " ", text) # remove all extra spaces 
    text = text.strip() # remove spaces at the start and at the end 
    text = re.sub(r"http\S+", "", text) # remove links 

    return text

In [21]:
ori_df = df.copy()
df.text = df.text.apply(clean_text)
print(f"{ori_df.text.values[0]}\n{df.text.values[0]}")

Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\band of ultra-cynics, are seeing green again.
wall st bears claw back black reuters reuters short sellers wall street dwindling band ultra cynics seeing green


In [22]:
df = df.dropna(subset=["label"]) 
df.head()

,text,label
0,wall st bears claw back black reuters reuters ...,2
1,carlyle looks toward commercial aerospace reut...,2
2,oil economy cloud stocks outlook reuters reute...,2
3,iraq halts oil exports main southern pipeline ...,2
4,oil prices soar time record posing new menace ...,2


In [23]:
label_decoder = {
    0: "World",
    1: "Sports",
    2: "Business",
    3: "Sci/Tech"
}

In [24]:
import numpy as np 
from transformers import BertTokenizer

In [ ]:
_tokenizer_cache = {}

def get_tokenizer(model_name="allenai/scibert_scivocab_uncased"):
    # cache per-process so map_batches workers don't reload the tokenizer on every batch
    if model_name not in _tokenizer_cache:
        _tokenizer_cache[model_name] = BertTokenizer.from_pretrained(model_name, return_dict=False)
    return _tokenizer_cache[model_name]

def tokenize(batch):
    tokenizer = get_tokenizer()
    encoded = tokenizer(batch["text"].tolist(), return_tensors="np", padding = "longest")
    return dict(ids=encoded["input_ids"], masks=encoded["attention_mask"], targets=np.array(batch["label"]))

In [26]:
tokenize(df.head(1))

{'ids': array([[  102,  3545,   177, 23309,  3895, 30128,  1542,  3778,   144,
         13342, 30113,   144, 13342, 30113,  2001, 27316,  3545, 10833,
         10029,   484,  2123,  2102, 10186, 27637,  1081, 18934,  3755,
           103]]),
 'masks': array([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1]]),
 'targets': array([2])}

In [27]:
# combining all preprocessing steps into function
def preprocess(df):
    df["text"] = df.text.apply(clean_text)
    targets = tokenize(df)
    return targets 

In [28]:
preprocess(df=train_df)

{'ids': array([[  102,   326,  2283, ...,     0,     0,     0],
        [  102, 19723, 15953, ...,     0,     0,     0],
        [  102,  3293,  1482, ...,     0,     0,     0],
        ...,
        [  102,   253, 17553, ...,     0,     0,     0],
        [  102, 23314, 30113, ...,     0,     0,     0],
        [  102,  3841,  2579, ...,     0,     0,     0]],
       shape=(96000, 199)),
 'masks': array([[1, 1, 1, ..., 0, 0, 0],
        [1, 1, 1, ..., 0, 0, 0],
        [1, 1, 1, ..., 0, 0, 0],
        ...,
        [1, 1, 1, ..., 0, 0, 0],
        [1, 1, 1, ..., 0, 0, 0],
        [1, 1, 1, ..., 0, 0, 0]], shape=(96000, 199)),
 'targets': array([1, 3, 0, ..., 3, 2, 3], shape=(96000,))}

In [29]:
from src.data import data, stratify_split
ray.data.DataContext.get_current().execution_options.preserve_order = True

2026-08-05 22:51:21,882	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


In [30]:
ds = ray.data.read_csv(dataset)
ds = ds.random_shuffle(seed=1234)
ds.take(1)


2026-08-05 22:51:25,018	INFO dataset.py:3818 -- Tip: Use `take_batch()` instead of `take() / show()` to return records in pandas or numpy batch format.
2026-08-05 22:51:25,034	INFO logging.py:416 -- Registered dataset logger for dataset dataset_2_0
2026-08-05 22:51:25,075	INFO streaming_executor.py:166 -- Starting execution of Dataset dataset_2_0. Full logs are in C:\Users\ASUS\AppData\Local\Temp\ray\session_2026-08-05_22-45-44_469526_16348\logs\ray-data
2026-08-05 22:51:25,076	INFO streaming_executor.py:167 -- Execution plan of Dataset dataset_2_0: InputDataBuffer[Input] -> TaskPoolMapOperator[ReadCSV] -> AllToAllOperator[RandomShuffle] -> LimitOperator[limit=1]
2026-08-05 22:51:25,085	INFO __init__.py:56 -- Progress will be logged because stdout is a non-interactive terminal.
2026-08-05 22:51:27,349	INFO logging_progress.py:174 -- ======= Running Dataset: dataset_2_0 =======
2026-08-05 22:51:27,352	INFO logging_progress.py:225 -- Total Progress: 0/?
2026-08-05 22:51:27,352	INFO loggi

[{'text': 'Microsoft timing is late for schools Microsoft #39;s decision to release a major upgrade for its flagship operating system in the same month that hundreds of thousands of students are reporting to college campuses across the nation is causing a major headache for some universities.',
  'label': 3}]

In [31]:
test_size = 0.2
train_ds, val_ds = stratify_split(ds, stratify="label", test_size=test_size)

In [32]:
label = train_ds.unique(column="label")
class_to_index = {label: i for i, label in enumerate(label)}

2026-08-05 22:52:02,961	INFO logging.py:416 -- Registered dataset logger for dataset dataset_10_0
2026-08-05 22:52:02,965	INFO hash_shuffle.py:1371 -- Estimated memory requirement for shuffling aggregator (partitions=200, aggregators=4, dataset (estimate)=0.0GiB): shuffle=6.9MiB, output=6.9MiB, total_base=13.8MiB, shuffle_aggregator_memory_estimate_skew_factor=1.3, total_with_skew=17.9MiB
2026-08-05 22:52:02,975	INFO streaming_executor.py:166 -- Starting execution of Dataset dataset_10_0. Full logs are in C:\Users\ASUS\AppData\Local\Temp\ray\session_2026-08-05_22-45-44_469526_16348\logs\ray-data
2026-08-05 22:52:02,978	INFO streaming_executor.py:167 -- Execution plan of Dataset dataset_10_0: InputDataBuffer[Input] -> TaskPoolMapOperator[ReadCSV] -> AllToAllOperator[RandomShuffle] -> HashShuffleOperator[Shuffle(key_columns=('label',), num_partitions=200)] -> AllToAllOperator[MapBatches(_add_split)->MapBatches(_filter_split)->RandomShuffle] -> HashAggregateOperator[HashAggregate(key_colu

In [33]:
sample_ds = train_ds.map_batches(preprocess, batch_format="pandas")
sample_ds.show(1)

2026-08-05 22:54:47,070	INFO logging.py:416 -- Registered dataset logger for dataset dataset_12_0
2026-08-05 22:54:47,074	INFO hash_shuffle.py:1371 -- Estimated memory requirement for shuffling aggregator (partitions=200, aggregators=4, dataset (estimate)=0.0GiB): shuffle=6.9MiB, output=6.9MiB, total_base=13.8MiB, shuffle_aggregator_memory_estimate_skew_factor=1.3, total_with_skew=17.9MiB
2026-08-05 22:54:47,080	INFO streaming_executor.py:166 -- Starting execution of Dataset dataset_12_0. Full logs are in C:\Users\ASUS\AppData\Local\Temp\ray\session_2026-08-05_22-45-44_469526_16348\logs\ray-data
2026-08-05 22:54:47,080	INFO streaming_executor.py:167 -- Execution plan of Dataset dataset_12_0: InputDataBuffer[Input] -> TaskPoolMapOperator[ReadCSV] -> AllToAllOperator[RandomShuffle] -> HashShuffleOperator[Shuffle(key_columns=('label',), num_partitions=200)] -> AllToAllOperator[MapBatches(_add_split)->MapBatches(_filter_split)->RandomShuffle] -> TaskPoolMapOperator[MapBatches(preprocess)] 

{'ids': array([  102, 17094, 20544,  3682,  8011, 18206, 14658, 15980,  8496,
         970, 11429,  3659,   107,  4641, 13465,   123,  1542, 26926,
         363,  9300,  3682,  8011,  2694,  5487,   303,   103,     0,
           0,     0,     0,     0,     0,     0,     0,     0,     0,
           0,     0,     0,     0,     0,     0,     0,     0,     0,
           0,     0,     0,     0,     0,     0,     0,     0,     0,
           0,     0,     0,     0,     0,     0,     0,     0,     0,
           0,     0,     0,     0,     0,     0,     0,     0,     0,
           0,     0,     0,     0,     0,     0,     0,     0,     0,
           0,     0,     0,     0,     0,     0,     0,     0,     0,
           0,     0,     0,     0,     0,     0,     0,     0,     0]), 'masks': array([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0

In [34]:
import os 
import random
import torch 
from ray.data.preprocessor import Preprocessor

def setting_seeds(seed=42):
    np.random.seed(seed)
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    eval("setattr(torch.backends.cudnn, 'deterministic', True)") # forces GPU to always perform calculation in exact sequence
    eval("setattr(torch.backends.cudnn, 'benchmark', False)") 
    os.environ["PYTHONHASHSEED"] = str(seed)

In [35]:
def loading_data(num_samples=None):
    ds = ray.data.read_csv(dataset)
    ds = ds.random_shuffle(seed=1234)
    ds = ray.data.from_items(ds.take(num_samples)) if num_samples else ds
    return ds

In [36]:
class CustomPreprocessor():
    """Custom Preprocess."""
    def __init__(self, label_decoder = None):
        self.label_decoder = label_decoder or {
            0: "World",
            1: "Sports",
            2: "Business",
            3: "Sci/Tech"
        }
        self.class_to_index = {v:k for k, v in self.label_decoder.items()}

    def fitting(self, ds):
        _ = ds.unique(column="label")
        return self
    
    def transforming(self, ds):
        return ds.map_batches(
            preprocess, 
            batch_format = "pandas")

In [37]:
import torch
import torch.nn as nn
from transformers import BertModel
from transformers import AutoTokenizer

In [38]:
model_name = "allenai/scibert_scivocab_uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
llm = BertModel.from_pretrained(model_name, return_dict = False)
embedding_dim = llm.config.hidden_size
num_classes = 4

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 24739.66it/s]
[transformers] BertModel LOAD REPORT from: allenai/scibert_scivocab_uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [39]:
class FinetunedLLM(nn.Module):
    def __init__(self, llm, dropout_p, embedding_dim, num_classes):
        super(FinetunedLLM, self).__init__()
        self.llm = llm
        self.dropout_p = dropout_p
        self.embedding_dim = embedding_dim
        self.num_classes = num_classes
        self.dropout = torch.nn.Dropout(dropout_p)
        self.fc1 =torch.nn.Linear(embedding_dim, num_classes)

    def forward(self, batch):
        ids, masks = batch["ids"], batch["masks"]
        seq, pool = self.llm(input_ids = ids, attention_mask = masks)
        z = self.dropout(pool)
        z = self.fc1(z)
        return z
    
    @torch.inference_mode()
    def predict(self,batch):
        self.eval()
        z = self(batch)
        y_pred = torch.argmax(z, dim=1).cpu().numpy()
        return y_pred
    
    def save(self, dp):
        with open(Path(dp, "args.json"), "w") as fp:
            contents = {
                "dropout_p": self.dropout_p,
                "embedding_dim": self.embedding_dim,
                "num_classes": self.num_classes,
            }
            json.dump(contents, fp, indent=4, sort_keys=False)
        torch.save(self.state_dict(), os.path.join(dp, "model.pt"))

    @classmethod
    def load(cls, args_fp, state_dict_fp):
        with open(args_fp, "r") as fp:
            kwargs = json.load(fp = fp)
        llm = BertModel.from_pretrained(model_name, return_dict = False)
        model = cls(llm = llm, **kwargs)
        model.load_state_dict(torch.load(state_dict_fp, map_location=torch.device("cpu")))
        return model 

In [40]:
model = FinetunedLLM(llm=llm, dropout_p=0.5, embedding_dim=embedding_dim, num_classes=num_classes)
print(model.named_parameters)

<bound method Module.named_parameters of FinetunedLLM(
  (llm): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(31090, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): Layer

In [41]:
from ray.train.torch import get_device

2026-08-05 22:58:14,434	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


In [42]:
def pad_array(arr, dtype=np.int32):
    max_len = max(len(row) for row in arr)
    padded_arr = np.zeros((arr.shape[0], max_len), dtype=dtype)
    for i, row in enumerate(arr):
        padded_arr[i][:len(row)] = row
    return padded_arr

In [43]:
def collate_fn(batch):
    batch["ids"] = pad_array(batch["ids"])
    batch["masks"] = pad_array(batch["masks"])
    dtypes = {"ids": torch.int32, "masks": torch.int32, "targets": torch.int64}
    tensor_batch = {}
    for key, array in batch.items():
        tensor_batch[key] = torch.as_tensor(array, dtype=dtypes[key], device=get_device())
    return tensor_batch

In [44]:
from pathlib import Path 
import ray.train as train
from ray.train import Checkpoint, CheckpointConfig, DataConfig, RunConfig, ScalingConfig
from ray.train.torch import TorchCheckpoint, TorchTrainer, TorchConfig
import tempfile
import torch.nn.functional as F
from torch.nn.parallel.distributed import DistributedDataParallel

In [45]:
def train_step(ds, batch_size, model, num_classes, loss_fn, optimizer):
    model.train()
    loss = 0.0
    ds_generator = ds.iter_torch_batches(batch_size=batch_size, collate_fn=collate_fn)
    for i, batch in enumerate(ds_generator):
        optimizer.zero_grad()
        z = model(batch)
        targets = F.one_hot(batch["targets"], num_classes=num_classes).float()
        J = loss_fn(z, targets)
        J.backward()
        optimizer.step()
        loss += (J.detach().item() - loss) / (i+1)
    return loss


In [46]:
def eval_step(ds, batch_size, model, num_classes, loss_fn):
    model.eval()
    loss = 0.0 
    y_trues, y_preds = [], []
    ds_generator = ds.iter_torch_batches(batch_size=batch_size, collate_fn=collate_fn)
    with torch.inference_mode():
        for i, batch in enumerate(ds_generator):
            z = model(batch)
            targets = F.one_hot(batch["targets"], num_classes=num_classes).float()
            J = loss_fn(z, targets).item()
            loss += (J-loss)/(i+1)
            y_trues.extend(batch["targets"].cpu().numpy())
            y_preds.extend(torch.argmax(z, dim=1).cpu().numpy())
        return loss, np.vstack(y_trues), np.vstack(y_preds)

In [47]:
def train_loop_per_worker(config):
    dropout_p = config["dropout_p"]
    lr = config["lr"]
    lr_factor = config["lr_factor"]
    lr_patience = config["lr_patience"]
    num_epochs = config["num_epochs"]
    batch_size = config["batch_size"]
    num_classes = config["num_classes"]

    setting_seeds()
    train_ds = train.get_dataset_shard("train")
    val_ds = train.get_dataset_shard("val")
    
    llm = BertModel.from_pretrained("allenai/scibert_scivocab_uncased", return_dict = False)
    model = FinetunedLLM(llm=llm, dropout_p=dropout_p, embedding_dim=llm.config.hidden_size, num_classes=num_classes)
    model = train.torch.prepare_model(model)

    loss_fn = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=lr_factor, patience=lr_patience)

    num_workers = train.get_context().get_world_size()
    batch_size_per_worker = batch_size // num_workers
    for epoch in range(num_epochs):
        train_loss = train_step(train_ds, batch_size_per_worker, model, num_classes, loss_fn, optimizer)
        val_loss, _, _ = eval_step(val_ds, batch_size_per_worker, model, num_classes, loss_fn)
        scheduler.step(val_loss)

        with tempfile.TemporaryDirectory() as dp:
            if isinstance(model, DistributedDataParallel):
                model.module.save(dp=dp)
            else:
                model.save(dp=dp)
            metrics = dict(epoch=epoch, lr=optimizer.param_groups[0]["lr"], train_loss=train_loss, val_loss=val_loss)
            checkpoint = Checkpoint.from_directory(dp)
            train.report(metrics, checkpoint=checkpoint)

In [48]:
from src.config import EFS_DIR, BASE_DIR, DATA_DIR, ARTIFACTS_DIR

In [49]:
train_loop_config = {
    "dropout_p": 0.5,
    "lr": 1e-4,
    "lr_factor": 0.8,
    "lr_patience": 3,
    "num_epochs": 10,
    "batch_size": 256,
    "num_classes": num_classes,
}

In [50]:
scaling_config = ScalingConfig(
    num_workers=num_workers,
    use_gpu=bool(resources_per_worker["GPU"]),
    resources_per_worker=resources_per_worker
)

In [51]:
checkpoint_config = CheckpointConfig(num_to_keep=1, checkpoint_score_attribute="val_loss", checkpoint_score_order="min")
run_config = RunConfig(name="llm", checkpoint_config=checkpoint_config, storage_path=EFS_DIR)

In [ ]:
ds = data(dataset_loc=dataset)
train_ds, val_ds = stratify_split(ds, stratify="label", test_size=test_size)

preprocessor = CustomPreprocessor()
preprocessor = preprocessor.fitting(train_ds)
# materialize now, while the full cluster is free, so trainer.fit() doesn't have to
# tokenize on the fly while competing with the training worker for CPU
train_ds = preprocessor.transforming(train_ds).materialize()
val_ds = preprocessor.transforming(val_ds).materialize()

In [ ]:
trainer = TorchTrainer(
    train_loop_per_worker=train_loop_per_worker,
    train_loop_config=train_loop_config,
    scaling_config=scaling_config,
    run_config=run_config,
    datasets={"train": train_ds, "val": val_ds},
    dataset_config=DataConfig(datasets_to_split=["train"]),
    torch_config=TorchConfig(backend="gloo"),
)
results = trainer.fit()
results

(raylet) Windows fatal exception: Windows fatal exception: access violation
(raylet) 
(raylet) access violation
(raylet) 
(TrainController pid=30192) Requesting resources: {'CPU': 3, 'GPU': 1} * 1
(TrainController pid=30192) Attempting to start training worker group of size 1 with the following resources: [{'CPU': 3, 'GPU': 1}] * 1
(RayTrainWorker pid=16772) Setting up process group for: env:// [rank=0, world_size=1]
(RayTrainWorker pid=16772) [W805 23:01:41.000000000 socket.cpp:759] [c10d] The client socket has failed to connect to [Eva]:63130 (system error: 10049 - The requested address is not valid in its context.).
(TrainController pid=30192) Started training worker group of size 1: 
(TrainController pid=30192) - (ip=127.0.0.1, pid=16772) world_rank=0, local_rank=0, node_rank=0
(RayTrainWorker pid=16772) HTTP Request: HEAD https://huggingface.co/allenai/scibert_scivocab_uncased/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
(RayTrainWorker pid=16772) Warning: You are se

SystemExit: 1

In [ ]:
best_checkpoint = results.best_checkpoints[0][0]
best_metrics = results.best_checkpoints[0][1]
print(f"best checkpoint: {best_checkpoint.path}")
print(f"best metrics: {best_metrics}")

## Stop now, upload epoch-0 checkpoint (closing session before training finishes)

Use this instead of the `results`/`best_checkpoint` path — `results` only exists if `trainer.fit()` has actually returned, which won't be true if you stop training early.

**Before running the next cell:** interrupt/stop the running training cell in this notebook. Safe to do — the epoch-0 checkpoint below is already fully written to disk regardless of whether the trainer loop keeps going.

Requires `HF_TOKEN` (write access) and `HF_REPO_ID` (e.g. `yourname/agnews-scibert-classifier`) set in `.env`.

In [ ]:
import os
from huggingface_hub import create_repo, upload_folder

# points directly at the epoch-0 checkpoint already saved to disk
# (currently the best val_loss of any epoch trained so far)
checkpoint_dir = str(EFS_DIR / "llm" / "checkpoint_2026-08-10_13-03-12.020315")

HF_REPO_ID = os.environ["HF_REPO_ID"]  # e.g. "yourname/agnews-scibert-classifier", set in .env

create_repo(HF_REPO_ID, exist_ok=True, repo_type="model", private=True)
upload_folder(
    folder_path=checkpoint_dir,
    repo_id=HF_REPO_ID,
    repo_type="model",
    commit_message="Upload epoch 0 checkpoint (best val_loss so far)",
)
print(f"Uploaded to https://huggingface.co/{HF_REPO_ID}")